In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
import joblib


In [ ]:
DATA_PATH = 'data.csv'
df = pd.read_csv(DATA_PATH)
df.shape


In [ ]:
df.head()


In [ ]:
df.info()


In [ ]:
df.isna().sum().sort_values(ascending=False)


In [ ]:
df = df.copy()
df['price'] = df['price'].str.replace(r'[$,]', '', regex=True).astype(float)
df['milage_num'] = df['milage'].str.extract(r'([0-9,]+)', expand=False).str.replace(',', '', regex=False).astype(float)
df['engine_hp'] = pd.to_numeric(df['engine'].str.extract(r'([0-9]+(?:\.[0-9]+)?)HP', expand=False), errors='coerce')
df['engine_liters'] = pd.to_numeric(df['engine'].str.extract(r'([0-9]+(?:\.[0-9]+)?)L', expand=False), errors='coerce')
df['engine_cylinders'] = pd.to_numeric(df['engine'].str.extract(r'(\d+) Cylinder', expand=False), errors='coerce')
df['clean_title'] = df['clean_title'].fillna('Unknown')
df = df.drop(columns=['milage'])
df = df.drop(columns=['price']).join(df['price'])
df.head()


In [ ]:
X = df.drop(columns=['price'])
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape


In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
numeric_features, categorical_features


In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', min_frequency=2))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])


In [ ]:
models = {
    'Ridge': Ridge(alpha=10.0),
    'RandomForest': RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1, min_samples_leaf=2),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=400, random_state=42, n_jobs=-1, min_samples_leaf=2)
}
results = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        'model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2': r2_score(y_test, pred)
    })
results_df = pd.DataFrame(results).sort_values('RMSE')
results_df


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    scores = -cross_val_score(pipe, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
    cv_results.append({'model': name, 'CV_RMSE_mean': scores.mean(), 'CV_RMSE_std': scores.std()})
pd.DataFrame(cv_results).sort_values('CV_RMSE_mean')


In [ ]:
best_name = results_df.iloc[0]['model']
best_model = Pipeline([('preprocessor', preprocessor), ('model', models[best_name])])
best_model.fit(X_train, y_train)
final_pred = best_model.predict(X_test)
final_metrics = pd.DataFrame([{
    'MAE': mean_absolute_error(y_test, final_pred),
    'RMSE': np.sqrt(mean_squared_error(y_test, final_pred)),
    'R2': r2_score(y_test, final_pred)
}])
final_metrics


In [ ]:
joblib.dump(best_model, 'eminence6_regression_model.joblib')
pd.DataFrame({'actual_price': y_test.values, 'predicted_price': final_pred}).to_csv('regression_predictions.csv', index=False)
